In [2]:
import pandas as pd
import numpy as np

# ── 输入输出路径 ────────────────────────────────────────────
IN_PATH     = "target_match_dataset.csv"
SKILLS_PATH = "agent_team_skills.csv"
OUT_PATH    = "data/match_players_labeled.csv"

# ── Carrier 判定阈值（队内 z-score）───────────────────────
# 必要条件：ACS z ≥ 1.5（综合得分显著高于队友）
# 附加条件：ADR / 击杀 / KAST 至少一项 z ≥ 对应阈值
CARRIER_ACS_Z   = 1.5
CARRIER_ADR_Z   = 1.0
CARRIER_KILLS_Z = 1.0
CARRIER_KAST_Z  = 1.0

# ── Loafer 判定阈值 ────────────────────────────────────────
LOAFER_BOTTOM_N     = 2    # 每队 ACS 倒数几名进入候选
LOAFER_AFK_ROUNDS   = 2    # afk_rounds ≥ N 触发 flag_afk
LOAFER_SPAWN_ROUNDS = 5    # rounds_in_spawn ≥ N 触发 flag_afk
LOAFER_FF_DAMAGE    = 100  # ff_outgoing ≥ N 触发 flag_ff
LOAFER_KAST_Z       = -1.5 # KAST 代理 z ≤ N 触发 flag_kast
LOAFER_SKILLS_Z     = -1.5 # 团队技能率 z ≤ N 触发 flag_skills
LOAFER_ADR_Z        = -1.0 # ADR z ≤ N 触发 flag_damage
LOAFER_MIN_FLAGS    = 2    # 至少满足几个维度才标记为 loafer

## Step 3：Carrier 与 Loafer 识别

本 notebook 实现对 `target_match_dataset.csv` 中每名玩家的角色打标，输出 `match_players_labeled.csv`。

### 整体流程
```
原始数据
  └─ 1. 计算衍生指标（ACS / ADR / KAST代理 / 团队技能使用率）
       └─ 2. 识别 Carrier（队内显著突出的玩家）
            └─ 3. 识别 Loafer（ACS倒数预筛选 + 多维度打标）
                 └─ 4. 保存带标签的数据集
                      └─ 5. 校验 + 共存分析
```

## 1. 工具函数

**组内 z-score**：Valorant 中每队 5 人，玩家的绝对数值（比如 ACS=200）意义有限，
更重要的是「相对于自己队友」的表现差异。
因此所有指标都在 `match_id + team` 组内做标准化，再判定是否显著高/低。

> z = (自己的值 - 队伍均值) / 队伍标准差
>
> z > 0 表示高于队伍平均，z < 0 表示低于队伍平均

In [3]:
def zscore_within(df, col, group=("match_id", "team")):
    """
    在 group 指定的组内计算 z-score。
    
    若组内标准差为 0（所有人数值相同，比如全队技能使用都是 0），
    则填 0（避免除以零报错，也表示「无差异」）。
    """
    grp = df.groupby(list(group))[col]
    mu  = grp.transform("mean")
    sd  = grp.transform("std").replace(0, np.nan)
    return ((df[col] - mu) / sd).fillna(0)

## 2. 衍生指标计算

原始数据中只有累计值（总得分、总伤害等），需要换算成「每轮」的指标，方便跨对局比较（因为不同对局轮次数不同）。

| 指标 | 公式 | 含义 |
|------|------|------|
| **ACS** | score / rounds_played | 官方综合表现指标，越高越好 |
| **ADR** | damage_made / rounds_played | 每轮平均输出伤害 |
| **KAST 代理** | (kills + assists + survived_rounds) / rounds_played | 近似官方 KAST（Kill/Assist/Survive/Trade）；无法计算 Trade，用存活轮数近似 |
| **团队技能使用率** | 团队技能总施放次数 / rounds_played | 结合 `agent_team_skills.csv`，只统计该 agent 对团队有价值的技能 |

> 注：KAST 代理值可能 > 1（玩家在同一轮既击杀又存活），作为相对比较指标使用，不影响 z-score 计算。

---

### Agent-level Baseline 校正

**动机**：不同 agent 在设计上就有系统性数值差距。以 ACS 为例，Reyna（纯决斗位）在打出同等影响力时，ACS 通常会显著高于 Sage（辅助位），因为治疗、护盾等辅助行为在 ACS 中权重较低。如果直接用原始值做队内 z-score 比较，一名优秀的 Sage 可能永远达不到 carrier 门槛，而一名摆烂的 Reyna 也可能因为 agent 本身数值膨胀而规避 loafer 标记，这在跨 agent 比较时引入了系统性偏差。

**校正方法**：对 `acs`、`adr`、`kast_proxy`、`team_skill_rate` 四个指标，分别计算该 agent 在**整个数据集**中的均值（agent-level baseline），再用原始值减去该均值，生成对应的 adjusted 列：

| 原始列 | Adjusted 列 | 含义 |
|--------|-------------|------|
| `acs` | `acs_adjusted` | 去除 agent 基准偏差后的综合得分 |
| `adr` | `adr_adjusted` | 去除 agent 基准偏差后的每轮伤害 |
| `kast_proxy` | `kast_adjusted` | 去除 agent 基准偏差后的团队贡献代理 |
| `team_skill_rate` | `skill_adjusted` | 去除 agent 基准偏差后的团队技能使用率 |

> **后续所有队内 z-score 比较均使用 adjusted 列**，以确保跨 agent 的公平比较。
> `kills` 不做校正，因为击杀数对 agent 类型的依赖性相对低，且保留原始击杀数有助于直觉理解。

In [4]:
def compute_metrics(df, skills_path=SKILLS_PATH):
    """
    在原始数据上添加所有派生指标列。
    
    团队技能使用率的计算逻辑：
      - 读取 agent_team_skills.csv，其中每个 agent 的 c/q/e/x_team 标注
        了哪些技能对团队有价值（1 = 团队技能，0 = 个人技能）
      - 将玩家的实际使用次数与该标注相乘，得到「团队技能施放总次数」
      - 除以 rounds_played 得到每轮使用率
      - 未在 CSV 中收录的新 agent 视为无团队技能（填 0）

    Agent-level baseline 校正：
      - 对 acs / adr / kast_proxy / team_skill_rate 各计算该 agent 在整个
        数据集中的均值，再用原始值减去该均值，生成 *_adjusted 列
      - 后续所有队内 z-score 比较均使用 adjusted 列，以消除不同 agent 之间
        的系统性数值差异（如 Reyna vs Sage 的固有 ACS 差距）
    """
    rp = df["rounds_played"].replace(0, np.nan)  # 防止除以零

    # 基础绩效指标
    df["acs"]        = df["score"]       / rp
    df["adr"]        = df["damage_made"] / rp
    df["afk_rate"]   = df["afk_rounds"]  / rp

    # KAST 代理：(击杀 + 助攻 + 存活轮数) / 总轮数
    survived = (rp - df["deaths"]).clip(lower=0)
    df["kast_proxy"] = (df["kills"] + df["assists"] + survived) / rp

    # 合并 agent 团队技能标注
    skills = pd.read_csv(skills_path)
    df = df.merge(skills, on="agent", how="left")
    for col in ["c_team", "q_team", "e_team", "x_team"]:
        df[col] = df[col].fillna(0)  # 未知 agent → 无团队技能

    # 团队技能施放次数 = 各技能实际使用次数 × 是否为团队技能
    df["team_skill_casts"] = (
        df["c_cast"] * df["c_team"] +
        df["q_cast"] * df["q_team"] +
        df["e_cast"] * df["e_team"] +
        df["x_cast"] * df["x_team"]
    )
    df["team_skill_slots"] = df[["c_team","q_team","e_team","x_team"]].sum(axis=1)
    df["team_skill_rate"]  = df["team_skill_casts"] / rp  # 每轮团队技能使用次数

    # 清理合并时产生的辅助列
    df = df.drop(columns=["c_team","q_team","e_team","x_team"])

    # ── Agent-level Baseline 校正 ──────────────────────────────────────
    # 对每个 agent 计算全数据集均值，再用原始值减去均值，消除 agent 固有的
    # 数值差异（如 Reyna 的 ACS 天然高于 Sage），让后续队内比较更公平。
    for raw_col, adj_col in [
        ("acs",            "acs_adjusted"),
        ("adr",            "adr_adjusted"),
        ("kast_proxy",     "kast_adjusted"),
        ("team_skill_rate","skill_adjusted"),
    ]:
        agent_mean = df.groupby("agent")[raw_col].transform("mean")
        df[adj_col] = df[raw_col] - agent_mean

    return df

## 3. Carrier 识别

**定义**：在己方 5 人队内，表现显著突出的玩家。

**判定逻辑**（队内 z-score 比较，基于 agent-level adjusted 值）：

```
ACS_adjusted z ≥ 1.5         ← 必要条件（去除 agent 偏差后综合得分显著高）
  AND
以下至少满足一项：
  ADR_adjusted   z ≥ 1.0     ← 每轮输出伤害显著高
  击杀数          z ≥ 1.0     ← 击杀数显著高（不做 baseline 校正）
  KAST_adjusted  z ≥ 1.0     ← 团队贡献度显著高
```

> **为什么需要附加条件？**
> ACS 包含了分辅助分、首杀分等多个来源，单纯 ACS 高有时是因为打了很多辅助。
> 附加一个实战维度（伤害/击杀/KAST）可以避免误标纯辅助位为 carrier。

> **为什么用 adjusted 值而非原始值？**
> 不同 agent 在设计上就有系统性数值差距（如 Reyna 的 ACS 天然高于 Sage）。
> 使用 agent-level baseline 校正后的值，确保 carrier 标记衡量的是「相对于同类型 agent 的超额贡献」，
> 而非 agent 本身的数值优势，从而实现跨 agent 的公平比较。

In [5]:
def label_carriers(df):
    """
    在 df 中新增以下列：
      z_acs, z_adr, z_kills, z_kast  — 各指标的队内 z-score
      is_carrier                     — 1 = carrier，0 = 普通

    z_acs / z_adr / z_kast 基于 agent-level baseline 校正后的 adjusted 列，
    以消除不同 agent 之间的系统性数值差异；z_kills 使用原始击杀数。
    """
    group = ("match_id", "team")

    df["z_acs"]   = zscore_within(df, "acs_adjusted",  group)
    df["z_adr"]   = zscore_within(df, "adr_adjusted",  group)
    df["z_kills"] = zscore_within(df, "kills",         group)
    df["z_kast"]  = zscore_within(df, "kast_adjusted", group)

    # ACS 显著高（必要条件）AND 至少一个其他维度也显著高（附加条件）
    df["is_carrier"] = (
        (df["z_acs"]   >= CARRIER_ACS_Z) &
        (
            (df["z_adr"]   >= CARRIER_ADR_Z)  |
            (df["z_kills"] >= CARRIER_KILLS_Z) |
            (df["z_kast"]  >= CARRIER_KAST_Z)
        )
    ).astype(int)

    n = df["is_carrier"].sum()
    print(f"Carrier：{n} 人（占 {n/len(df):.1%}）")
    return df

## 4. Loafer 识别

**定义**：在团队中有明显「摆烂」或「社会懈怠」行为的玩家。

### 两步走设计

**步骤 1 — 预筛选（缩小候选范围）**
每队 5 人中 **ACS_adjusted 倒数 2 名**进入候选。
使用 baseline 校正后的 ACS 进行排名，避免辅助位 agent（如 Sage、Skye）因 ACS 天然偏低而被系统性地过度筛入候选，同时也防止高 ACS agent（如 Reyna）的摆烂玩家凭借 agent 数值优势规避预筛选。

**步骤 2 — 多维度打标（5 个维度，满足 ≥2 个 = loafer）**

| Flag | 触发条件 | 数据来源 | 是否使用 baseline 校正 |
|------|----------|----------|------------------------|
| `flag_afk` | afk_rounds ≥ 2 或 rounds_in_spawn ≥ 5 | 最直观的不参与游戏行为 | ✗ 纯行为指标，与 agent 无关 |
| `flag_ff` | ff_outgoing ≥ 100 | 大量伤害队友（主动捣乱）| ✗ 纯行为指标，与 agent 无关 |
| `flag_kast` | **KAST_adjusted** z ≤ -1.0 | 去除 agent 偏差后几乎不为团队做贡献 | ✓ |
| `flag_skills` | **skill_adjusted** z ≤ -1.0（且 agent ≥ 2 个团队技能槽）| 不履行 agent 的团队职责 | ✓ |
| `flag_damage` | **ADR_adjusted** z ≤ -1.0 | 去除 agent 偏差后输出伤害显著低 | ✓ |

> `flag_afk` 和 `flag_ff` 保持原始值：这两个是纯行为指标（挂机/伤害队友），
> 与所使用的 agent 无关，不需要也不应该做 baseline 校正。
>
> `flag_skills` 附加了「agent ≥ 2 个团队技能槽」的条件，
> 是因为纯决斗位（如 Reyna/Jett）本身团队技能少，用绝对使用率不公平。

In [6]:
def label_loafers(df):
    """
    在 df 中新增以下列：
      acs_rank      — 队内 ACS_adjusted 排名（1 = 最低）
      is_bottom_acs — 是否在 ACS_adjusted 倒数 LOAFER_BOTTOM_N 名内
      flag_*        — 各维度摆烂标志（0/1）
      loafing_flags — 总 flag 数（0~5）
      is_loafer     — 1 = 高度疑似摆烂，0 = 正常

    预筛选和 flag_kast / flag_damage / flag_skills 均使用 agent-level
    baseline 校正后的 adjusted 列；flag_afk 和 flag_ff 是纯行为指标，
    与 agent 无关，保持原样。
    """
    group = ("match_id", "team")

    # ── 步骤 1：ACS_adjusted 倒数预筛选 ───────────────────────
    # 使用 baseline 校正后的 ACS 排名，避免辅助位 agent 被系统性过度筛入，
    # 也防止高 ACS agent 的摆烂玩家凭 agent 数值优势规避预筛选。
    # rank 从 1 开始，1 = 队内 acs_adjusted 最低
    df["acs_rank"]      = df.groupby(list(group))["acs_adjusted"].rank(method="min", ascending=True)
    df["is_bottom_acs"] = (df["acs_rank"] <= LOAFER_BOTTOM_N).astype(int)

    # ── 步骤 2：补算 z-score（若 label_carriers 未算过则补算）
    if "z_adr"  not in df.columns: df["z_adr"]  = zscore_within(df, "adr_adjusted",  group)
    if "z_kast" not in df.columns: df["z_kast"] = zscore_within(df, "kast_adjusted", group)
    df["z_skills"] = zscore_within(df, "skill_adjusted", group)

    # ── 步骤 3：各维度 flag ────────────────────────────────────

    # Flag 1: 明显不参与游戏（挂机 / 长时间蹲 spawn）— 纯行为指标，不做校正
    df["flag_afk"] = (
        (df["afk_rounds"]      >= LOAFER_AFK_ROUNDS)   |
        (df["rounds_in_spawn"] >= LOAFER_SPAWN_ROUNDS)
    ).astype(int)

    # Flag 2: 大量伤害队友（主动摆烂 / 捣乱）— 纯行为指标，不做校正
    df["flag_ff"] = (df["ff_outgoing"] >= LOAFER_FF_DAMAGE).astype(int)

    # Flag 3: KAST_adjusted 显著低（去除 agent 偏差后几乎不为团队做贡献）
    df["flag_kast"] = (df["z_kast"] <= LOAFER_KAST_Z).astype(int)

    # Flag 4: skill_adjusted 使用率显著低
    df["flag_skills"] = (
        (df["z_skills"]<= LOAFER_SKILLS_Z) 
    ).astype(int)

    # Flag 5: ADR_adjusted 显著低（去除 agent 偏差后输出伤害显著低）
    df["flag_damage"] = (df["z_adr"] <= LOAFER_ADR_Z).astype(int)

    # ── 步骤 4：综合判定 ───────────────────────────────────────
    flag_cols = ["flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage"]
    df["loafing_flags"] = df[flag_cols].sum(axis=1)

    # Loafer = 在 ACS_adjusted 倒数候选名单内 AND 满足 ≥ LOAFER_MIN_FLAGS 个维度
    df["is_loafer"] = (
        (df["is_bottom_acs"]  == 1) &
        (df["loafing_flags"] >= LOAFER_MIN_FLAGS)
    ).astype(int)

    n = df["is_loafer"].sum()
    print(f"Loafer：{n} 人（占 {n/len(df):.1%}）")
    return df

## 5. 主流程

将以上函数串联起来：读取数据 → 计算指标 → 打标签 → 保存。

In [7]:
def build_labeled(in_path=IN_PATH, out_path=OUT_PATH):
    """
    主流程：
      输入：target_match_dataset.csv
      输出：data/match_players_labeled.csv（新增 carrier/loafer 标签）
    """
    print("读取原始数据...")
    df = pd.read_csv(in_path).drop_duplicates(subset=["match_id", "puuid"])
    print(f"  {df.shape[0]:,} 行 × {df.shape[1]} 列")

    print("\n计算衍生指标...")
    df = compute_metrics(df)

    print("\n识别 Carrier...")
    df = label_carriers(df)

    print("\n识别 Loafer...")
    df = label_loafers(df)

    df.to_csv(out_path, index=False)
    print(f"\n✅ 保存完成 → {out_path}  shape={df.shape}")
    return df

## 6. 合理性校验

打完标签后需要验证结果是否符合预期：

- **Carrier 比例**：Valorant 5v5 中 carrier 应该是少数（预期约 5%~15%）
- **Loafer 比例**：摆烂玩家也应是少数（预期约 5%~20%）
- **Carrier 与 Loafer 不应重叠**：同一玩家不可能同时是 carrier 和 loafer
- **Carrier 的各项指标均值应显著高于普通玩家**
- **Loafer 的各项指标均值应显著低于普通玩家**

In [8]:
def sanity_check(df):
    """对打标结果进行合理性校验，输出各项统计信息。"""
    n   = len(df)
    n_c = df["is_carrier"].sum()
    n_l = df["is_loafer"].sum()
    both = ((df["is_carrier"] == 1) & (df["is_loafer"] == 1)).sum()

    print("=" * 55)
    print("合理性校验")
    print("=" * 55)
    print(f"总行数：      {n:,}")
    print(f"Carrier：     {n_c}（{n_c/n:.1%}）")
    print(f"Loafer：      {n_l}（{n_l/n:.1%}）")
    print(f"两者重叠：    {both}（应为 0）")

    cols = ["acs", "adr", "kills", "kast_proxy"]
    print("\n── Carrier 均值 vs 普通玩家 ──")
    print(df.groupby("is_carrier")[cols].mean().round(1).to_string())

    print("\n── Loafer 均值 vs 普通玩家 ──")
    extra = ["afk_rate", "loafing_flags"]
    print(df.groupby("is_loafer")[cols + extra].mean().round(3).to_string())

    print("\n── 每队 Carrier 数量分布 ──")
    print(df.groupby(["match_id","team"])["is_carrier"].sum()
            .value_counts().sort_index().to_string())

    print("\n── 每队 Loafer 数量分布 ──")
    print(df.groupby(["match_id","team"])["is_loafer"].sum()
            .value_counts().sort_index().to_string())

    print("\n── loafing_flags 分布（0~5）──")
    print(df["loafing_flags"].value_counts().sort_index().to_string())
    print("=" * 55)

## 7. 共存分析

**核心研究问题**：队伍里有 carrier 时，其他队友的懈怠程度是否更高？

**方法**：
1. 计算每队是否存在 carrier（`team_has_carrier`）
2. 只看非 carrier 的队友（carrier 本人排除出分析）
3. 比较「有 carrier 的队伍」vs「无 carrier 的队伍」中，队友的 loafing_flags 和 is_loafer 比例

这是 Research Question 1 的初步答案。

In [9]:
def coexistence_check(df):
    """
    分析有/无 carrier 的队伍中，非 carrier 队友的懈怠指标对比。
    返回 teammates DataFrame（非 carrier 子集），供后续深入分析使用。
    """
    # 计算每队是否存在 carrier
    team_has_carrier = (
        df.groupby(["match_id", "team"])["is_carrier"]
          .max()
          .rename("team_has_carrier")
          .reset_index()
    )
    df = df.merge(team_has_carrier, on=["match_id", "team"])

    # 只分析非 carrier 的队友
    teammates = df[df["is_carrier"] == 0].copy()

    print("=" * 55)
    print("共存分析：有/无 Carrier 队伍中的非 Carrier 队友")
    print("=" * 55)

    print("\n── loafing_flags 均值（越高 = 懈怠迹象越多）──")
    print(teammates.groupby("team_has_carrier")["loafing_flags"].mean().round(3).to_string())

    print("\n── 各维度 flag 触发率 ──")
    flag_cols = ["flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage"]
    print(teammates.groupby("team_has_carrier")[flag_cols].mean().round(3).to_string())

    print("\n── is_loafer 比例 ──")
    print(teammates.groupby("team_has_carrier")["is_loafer"].mean().round(3).to_string())

    # 同时有 carrier 和 loafer 的队伍（「共存局」）
    coexist = (
        df.groupby(["match_id", "team"])
          .agg(n_carrier=("is_carrier","sum"), n_loafer=("is_loafer","sum"))
          .query("n_carrier >= 1 and n_loafer >= 1")
          .reset_index()
    )
    print(f"\n共存局（同队有 carrier + loafer）：{len(coexist)} 队")

    # 随机抽 3 个共存局展示明细（含各维度 flag）
    print("\n── 随机抽取 3 个共存局明细 ──")
    show = ["name", "acs","agent", "kills", "afk_rounds",
            "flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage",
            "loafing_flags", "is_carrier", "is_loafer"]
    for _, row in coexist.sample(min(3, len(coexist)), random_state=42).iterrows():
        sub = df[(df["match_id"] == row["match_id"]) & (df["team"] == row["team"])]
        print(f"\nmatch={row['match_id'][:12]}  team={row['team']}")
        print(sub[show].to_string(index=False))

    return teammates

## 8. 执行

In [10]:
df = build_labeled()
sanity_check(df)
teammates = coexistence_check(df)

读取原始数据...
  14,810 行 × 37 列

计算衍生指标...

识别 Carrier...
Carrier：745 人（占 5.0%）

识别 Loafer...
Loafer：498 人（占 3.4%）

✅ 保存完成 → data/match_players_labeled.csv  shape=(14810, 63)
合理性校验
总行数：      14,810
Carrier：     745（5.0%）
Loafer：      498（3.4%）
两者重叠：    0（应为 0）

── Carrier 均值 vs 普通玩家 ──
              acs    adr  kills  kast_proxy
is_carrier                                 
0           206.0  135.9   15.4         1.2
1           322.1  205.6   24.3         1.7

── Loafer 均值 vs 普通玩家 ──
               acs      adr   kills  kast_proxy  afk_rate  loafing_flags
is_loafer                                                               
0          214.785  141.264  16.117       1.269     0.000          0.163
1          128.536   86.482   9.378       0.878     0.001          2.062

── 每队 Carrier 数量分布 ──
is_carrier
0    2217
1     745

── 每队 Loafer 数量分布 ──
is_loafer
0    2464
1     498

── loafing_flags 分布（0~5）──
loafing_flags
0    11979
1     2331
2      469
3       31
共存分析：有/无 Carrier 队伍中的非 Carrier 队

In [11]:
df.columns

Index(['match_id', 'puuid', 'name', 'tag', 'team', 'region', 'map', 'patch',
       'rounds_played', 'game_start', 'rank_tier', 'rank_patched', 'agent',
       'player_level', 'score', 'kills', 'deaths', 'assists', 'damage_made',
       'damage_received', 'headshot_rate', 'headshots', 'bodyshots',
       'legshots', 'c_cast', 'q_cast', 'e_cast', 'x_cast',
       'economy_spent_overall', 'economy_spent_avg', 'loadout_value_overall',
       'loadout_value_avg', 'afk_rounds', 'rounds_in_spawn', 'ff_outgoing',
       'ff_incoming', 'team_won', 'acs', 'adr', 'afk_rate', 'kast_proxy',
       'team_skill_casts', 'team_skill_slots', 'team_skill_rate',
       'acs_adjusted', 'adr_adjusted', 'kast_adjusted', 'skill_adjusted',
       'z_acs', 'z_adr', 'z_kills', 'z_kast', 'is_carrier', 'acs_rank',
       'is_bottom_acs', 'z_skills', 'flag_afk', 'flag_ff', 'flag_kast',
       'flag_skills', 'flag_damage', 'loafing_flags', 'is_loafer'],
      dtype='str')

In [12]:
df[(df['flag_skills'] == 1)&(df['is_loafer'] == 1)][['agent','acs','e_cast','rounds_played','q_cast','x_cast','c_cast','z_skills','flag_skills','flag_kast','flag_damage']].head(20)

,agent,acs,e_cast,rounds_played,q_cast,x_cast,c_cast,z_skills,flag_skills,flag_kast,flag_damage
2,Reyna,77.863636,0.0,22,4.0,1.0,5.0,-1.610522,1,0,1
210,Skye,130.217391,11.0,23,13.0,2.0,7.0,-1.568414,1,0,1
227,Sova,116.600000,13.0,15,0.0,1.0,6.0,-1.571278,1,0,1
335,Fade,116.523810,16.0,21,5.0,2.0,17.0,-1.672302,1,0,1
511,Skye,157.384615,20.0,26,9.0,1.0,1.0,-1.664136,1,1,0
531,Chamber,116.043478,11.0,23,6.0,2.0,4.0,-1.594488,1,1,1
550,Clove,89.833333,16.0,18,0.0,1.0,2.0,-1.779943,1,1,1
852,Raze,216.100000,8.0,20,16.0,2.0,5.0,-1.527712,1,0,1
1015,Miks,106.380952,35.0,21,6.0,2.0,7.0,-1.650904,1,0,1
1077,Phoenix,184.217391,17.0,23,4.0,5.0,2.0,-1.759161,1,0,1


In [13]:
df.iloc[173,:]

match_id         fd98621e-ef14-439b-b480-0a0651bb8a43
puuid            b06ac4d9-0c60-5363-9b33-9b03d391d5b8
name                                             Domo
tag                                              0121
team                                             Blue
                                 ...                 
flag_kast                                           0
flag_skills                                         1
flag_damage                                         0
loafing_flags                                       1
is_loafer                                           0
Name: 173, Length: 63, dtype: object

In [14]:
match_table = pd.read_csv("target_match_dataset.csv")

match_table.columns

Index(['match_id', 'puuid', 'name', 'tag', 'team', 'region', 'map', 'patch',
       'rounds_played', 'game_start', 'rank_tier', 'rank_patched', 'agent',
       'player_level', 'score', 'kills', 'deaths', 'assists', 'damage_made',
       'damage_received', 'headshot_rate', 'headshots', 'bodyshots',
       'legshots', 'c_cast', 'q_cast', 'e_cast', 'x_cast',
       'economy_spent_overall', 'economy_spent_avg', 'loadout_value_overall',
       'loadout_value_avg', 'afk_rounds', 'rounds_in_spawn', 'ff_outgoing',
       'ff_incoming', 'team_won'],
      dtype='str')

In [15]:
loafer_rate_by_agent = df.groupby('agent')['is_loafer'].mean()
display(loafer_rate_by_agent.sort_values(ascending=False))

agent
Brimstone    0.059091
Miks         0.051342
Veto         0.050633
Killjoy      0.048443
Vyse         0.048000
Fade         0.045045
Tejo         0.044534
Phoenix      0.040767
Neon         0.040419
Gekko        0.037736
Yoru         0.037037
Waylay       0.036313
Astra        0.036036
Sova         0.034956
Clove        0.033445
Omen         0.031977
Skye         0.031720
Breach       0.030928
Chamber      0.030900
Reyna        0.030815
Deadlock     0.024390
Raze         0.023372
Iso          0.022599
Harbor       0.022222
Cypher       0.021021
Jett         0.020313
Viper        0.019934
Sage         0.018182
KAY/O        0.000000
Name: is_loafer, dtype: float64

In [16]:
from IPython.display import display

# 设置显示选项，不截断列和行
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

# 捞出所有共存局
coexist_teams = (
    df.groupby(["match_id", "team"])
      .agg(n_carrier=("is_carrier", "sum"), n_loafer=("is_loafer", "sum"))
      .query("n_carrier >= 1 and n_loafer >= 1")
      .reset_index()
)

coexist_full = df.merge(coexist_teams[["match_id", "team"]], on=["match_id", "team"])

# 按 match_id 和 team 排序，方便逐场看
coexist_full = coexist_full.sort_values(["match_id", "team"])

print(f"共存局：{coexist_teams.shape[0]} 队，{coexist_full.shape[0]} 行")
display(coexist_full)

共存局：30 队，150 行


,match_id,puuid,name,tag,team,region,map,patch,rounds_played,game_start,rank_tier,rank_patched,agent,player_level,score,kills,deaths,assists,damage_made,damage_received,headshot_rate,headshots,bodyshots,legshots,c_cast,q_cast,e_cast,x_cast,economy_spent_overall,economy_spent_avg,loadout_value_overall,loadout_value_avg,afk_rounds,rounds_in_spawn,ff_outgoing,ff_incoming,team_won,acs,adr,afk_rate,kast_proxy,team_skill_casts,team_skill_slots,team_skill_rate,acs_adjusted,adr_adjusted,kast_adjusted,skill_adjusted,z_acs,z_adr,z_kills,z_kast,is_carrier,acs_rank,is_bottom_acs,z_skills,flag_afk,flag_ff,flag_kast,flag_skills,flag_damage,loafing_flags,is_loafer
0,016304a4-d3f8-4beb-8f2c-8bffc1433e6a,0ec9a69a-14de-53d8-ae8e-7c9a7b5e0f72,목탁치다걸린예수님,1111,Red,kr,Split,release-12.06-shipping-19-4440219,17,1775988329,22,Ascendant 2,Skye,729,2520,9,10,6,1604,2086,0.1379,4,20,5,2.0,11.0,28.0,1.0,45300,2664.7058,70150,4126.4707,0.0,0.0,0.0,0.0,True,148.235294,94.352941,0.000,1.294118,42.0,4.0,2.470588,-41.264644,-29.029623,-0.015725,0.356023,-0.765310,-1.011597,-0.732686,-1.501679,0,1.0,1,-0.461418,0,0,1,0,1,2,1
1,016304a4-d3f8-4beb-8f2c-8bffc1433e6a,cc09c96f-9386-595b-8dec-b23c9d9ac12a,이쁜이는 뒤를 돌아봅니다,오 버,Red,kr,Split,release-12.06-shipping-19-4440219,17,1775988329,20,Diamond 3,Sage,708,3100,10,5,9,2000,1003,0.1818,4,17,1,12.0,24.0,8.0,1.0,39300,2311.7646,77300,4547.0586,0.0,0.0,0.0,0.0,True,182.352941,117.647059,0.000,1.823529,45.0,4.0,2.647059,-7.272515,-6.603079,0.569922,0.804950,-0.459284,-0.534479,-0.618204,0.289073,0,3.0,0,0.592755,0,0,0,0,0,0,0
2,016304a4-d3f8-4beb-8f2c-8bffc1433e6a,383f0b26-8eff-5716-9ab2-267535867a5a,おこのみやき,맛있어,Red,kr,Split,release-12.06-shipping-19-4440219,17,1775988329,21,Ascendant 1,Miks,465,3338,11,7,14,2255,1340,0.1538,6,33,0,20.0,14.0,37.0,2.0,42350,2491.1765,71900,4229.4116,0.0,0.0,0.0,0.0,True,196.352941,132.647059,0.000,2.058824,73.0,4.0,4.294118,-8.186544,2.899887,0.453019,1.100488,-0.467513,-0.332306,-0.503721,-0.068383,0,2.0,1,1.286737,0,0,0,0,0,0,0
3,016304a4-d3f8-4beb-8f2c-8bffc1433e6a,ab488009-887d-5038-8401-9173bc737a2b,알리페데,조 한,Red,kr,Split,release-12.06-shipping-19-4440219,17,1775988329,22,Ascendant 2,Neon,350,7722,30,14,2,3921,2665,0.1786,25,99,16,10.0,6.0,17.0,2.0,48800,2870.5881,61850,3638.2354,0.0,0.0,0.0,0.0,True,454.235294,230.647059,0.000,2.058824,16.0,2.0,0.941176,235.604186,92.697123,0.896323,-0.014449,1.727296,1.578103,1.671439,1.287122,1,5.0,0,-1.331361,0,0,0,0,0,0,0
4,016304a4-d3f8-4beb-8f2c-8bffc1433e6a,cb1d17ae-22fb-5018-bbcf-07ebb1a4bb53,고민중,텅텅빔,Red,kr,Split,release-12.06-shipping-19-4440219,17,1775988329,20,Diamond 3,Jett,189,4496,17,9,3,3058,2054,0.3415,14,24,3,18.0,7.0,18.0,1.0,40450,2379.4119,64150,3773.5293,0.0,0.0,0.0,0.0,True,264.470588,179.882353,0.000,1.647059,18.0,1.0,1.058824,39.834158,32.634075,0.473377,0.515595,-0.035190,0.300280,0.183171,-0.006133,0,4.0,0,-0.086712,0,0,0,0,0,0,0
70,04b725eb-3e1e-4d67-a6ed-b8fae029fad8,69d02afb-a203-5177-a77f-b1caec5b4570,zuichy,Lei,Blue,na,Bind,release-12.05-shipping-26-4440267,26,1774911315,24,Immortal 1,Chamber,627,5285,18,21,3,3552,3534,0.2188,7,22,3,27.0,8.0,16.0,3.0,66400,2553.8462,104100,4003.8462,0.0,0.0,0.0,0.0,False,203.269231,136.615385,0.000,1.000000,27.0,1.0,1.038462,-10.785699,-4.706515,-0.194193,0.077863,-0.057469,-0.035647,0.069317,-0.273475,0,3.0,0,0.571002,0,0,0,0,0,0,0
71,04b725eb-3e1e-4d67-a6ed-b8fae029fad8,a4f447ca-1dbb-5c5b-9c70-2de071a18dd2,Atheros,Mommy,Blue,na,Bind,release-12.05-shipping-26-4440267,26,1774911315,24,Immortal 1,Miks,450,5688,17,16,16,3561,3191,0.4878,20,20,1,15.0,20.0,49.0,3.0,68400,2630.7693,101550,3905.7693,0.0,0.0,0.0,0.0,False,218.769231,136.961538,0.000,1.653846,87.0,4.0,3.346154,14.229746,7.214367,0.048042,0.152524,0.271748,0.251161,-0.103975,0.679392,0,4.0,0,0.928538,0,0,0,0,0,0,0
72,04b725eb-3e1e-4d67-a6ed-b8fae029fad8,aa254376-b0f5-59a8-ac5f-9163311183f5,Byzoz deus,byzoz,Blue,na,Bind,release-12.05-shipping-26-4440267,26,1774911315,23,Ascendant 3,Raze,308,4318,14,16,5,3457,3418,0.2344,15,43,6,13.0,